# Flight Delay Analytics — Integrated Exploratory Analysis

**Author:** Daniel Vargas
**Date:** July 2026
**Dataset:** BTS On-Time Performance 2024 + NOAA/IEM ASOS (historical weather)
**Scope:** Top 30 U.S. airports by traffic, full 2024 calendar year

**Business objective:** What factors best predict operational flight delays?

## 0. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

## 1. Data Context and Loading

This notebook loads the **clean, deduplicated** dataset produced by `01_eda.ipynb` (Section 1). For the full integration rationale — BTS scope, NOAA/IEM merge logic, timezone handling, weather deduplication, and BTS duplicate removal — see **`01_eda.ipynb`**.

**Key facts about the dataset:**
- **4,593,373 rows** × 47 columns (after removing ~1,044 BTS duplicate records and ~110 DST-transition rows)
- **Scope:** flights departing from a top-30 U.S. airport (delay measured at origin)
- **Origin weather:** available for 100% of rows
- **Destination weather:** available for 55.8% of rows (only when destination is also a top-30 station)
- **Format:** Parquet (column-typed, ~60% smaller than CSV)

In [ ]:
df = pd.read_parquet('../data/processed/flights_with_weather.parquet')
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

## 2. Initial Exploration

_Data types, dimensions, nulls, duplicates._

In [ ]:
df.info()

In [ ]:
# Null summary — useful to identify which weather columns have incomplete coverage
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
null_df = pd.DataFrame({'nulls': null_counts, 'pct': null_pct})
null_df[null_df['nulls'] > 0].sort_values('pct', ascending=False)

In [ ]:
# Sanity check: verify no duplicates remain (dedup was done in 01_eda before saving)
remaining = df.duplicated(
    subset=['FlightDate', 'Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime'],
    keep=False
).sum()
print(f"Duplicate rows: {remaining:,}" if remaining else "No duplicates — clean.")

## 3. Descriptive Statistics

_Central tendency, dispersion, and position — broken down by airline, airport, and time block. Interpret each metric in business context._

### 3.0 Feature Engineering for Breakdowns

Before computing metrics, we derive three grouping variables needed by the rubric (airline, airport, time block) plus season, since seasonality is central to the business question:

- **`dep_time_block`**: scheduled departure hour bucketed into Morning/Afternoon/Evening/Night. Built from `CRSDepTime` (scheduled, not actual) so it reflects planned operations, not a delay-inflated version of the hour.
- **`season`**: derived from `Month`, using standard Northern Hemisphere seasons (matches the business context — U.S. airports).

`Reporting_Airline` and `Origin` already exist in the dataset and are used directly for the airline/airport breakdowns.

In [ ]:
# --- 3.0 Feature engineering: time block and season ---

def hour_from_crs(x):
    """Extract the scheduled hour (0-23) from HHMM-style CRSDepTime, handling the '2400' edge case."""
    x = int(x)
    return (x // 100) % 24

df['dep_hour'] = df['CRSDepTime'].apply(hour_from_crs)

def time_block(h):
    if 5 <= h < 12:
        return 'Morning (05-11)'
    elif 12 <= h < 17:
        return 'Afternoon (12-16)'
    elif 17 <= h < 21:
        return 'Evening (17-20)'
    else:
        return 'Night (21-04)'

df['dep_time_block'] = df['dep_hour'].apply(time_block)

season_map = {12: 'Winter', 1: 'Winter', 2: 'Winter',
              3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer',
              9: 'Fall', 10: 'Fall', 11: 'Fall'}
df['season'] = df['Month'].map(season_map)

print(df[['dep_hour', 'dep_time_block', 'season']].head())
print()
print(df['dep_time_block'].value_counts())
print()
print(df['season'].value_counts())

### 3.1 Central Tendency, Dispersion, and Position Metrics

We compute metrics on both delay variables:
- **`DepDelay` / `ArrDelay`**: signed delay in minutes (negative = early). Useful for understanding the full distribution, including how often flights leave early.
- **`DepDelayMinutes` / `ArrDelayMinutes`**: delay clipped at 0 (BTS convention — early departures count as 0, not negative). This is the operationally relevant variable for "how late," so it's the one used in the breakdowns below.

Metrics: mean, median, std, and position metrics (p25, p75, p90, p95, p99) — the upper percentiles matter more here than a symmetric spread, since delay distributions are right-skewed (most flights are on time or slightly early; a long tail drives the average up).

In [ ]:
# --- 3.1 Overall descriptive statistics ---

delay_cols = ['DepDelay', 'DepDelayMinutes', 'ArrDelay', 'ArrDelayMinutes']

def describe_delay(series):
    return pd.Series({
        'count': series.count(),
        'mean': series.mean(),
        'median': series.median(),
        'std': series.std(),
        'p25': series.quantile(0.25),
        'p75': series.quantile(0.75),
        'p90': series.quantile(0.90),
        'p95': series.quantile(0.95),
        'p99': series.quantile(0.99),
        'max': series.max(),
    })

overall_stats = df[delay_cols].apply(describe_delay).T
overall_stats.round(1)

In [ ]:
# --- 3.1 Breakdown by airline ---

airline_stats = df.groupby('Reporting_Airline').agg(
    n_flights=('DepDelayMinutes', 'count'),
    mean_delay=('DepDelayMinutes', 'mean'),
    median_delay=('DepDelayMinutes', 'median'),
    std_delay=('DepDelayMinutes', 'std'),
    p90_delay=('DepDelayMinutes', lambda x: x.quantile(0.90)),
    p99_delay=('DepDelayMinutes', lambda x: x.quantile(0.99)),
    pct_delayed_15=('DepDel15', lambda x: x.mean() * 100),
).round(1).sort_values('mean_delay', ascending=False)

airline_stats

In [ ]:
# --- 3.1 Breakdown by airport (origin) ---

airport_stats = df.groupby('Origin').agg(
    n_flights=('DepDelayMinutes', 'count'),
    mean_delay=('DepDelayMinutes', 'mean'),
    median_delay=('DepDelayMinutes', 'median'),
    std_delay=('DepDelayMinutes', 'std'),
    p90_delay=('DepDelayMinutes', lambda x: x.quantile(0.90)),
    p99_delay=('DepDelayMinutes', lambda x: x.quantile(0.99)),
    pct_delayed_15=('DepDel15', lambda x: x.mean() * 100),
).round(1).sort_values('mean_delay', ascending=False)

airport_stats

In [ ]:
# --- 3.1 Breakdown by time block ---

timeblock_order = ['Morning (05-11)', 'Afternoon (12-16)', 'Evening (17-20)', 'Night (21-04)']

timeblock_stats = df.groupby('dep_time_block').agg(
    n_flights=('DepDelayMinutes', 'count'),
    mean_delay=('DepDelayMinutes', 'mean'),
    median_delay=('DepDelayMinutes', 'median'),
    std_delay=('DepDelayMinutes', 'std'),
    p90_delay=('DepDelayMinutes', lambda x: x.quantile(0.90)),
    p99_delay=('DepDelayMinutes', lambda x: x.quantile(0.99)),
    pct_delayed_15=('DepDel15', lambda x: x.mean() * 100),
).round(1).reindex(timeblock_order)

timeblock_stats

In [ ]:
# --- 3.1 Visualization: mean delay by airline and by time block ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

airline_stats['mean_delay'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Mean Departure Delay by Airline (min)')
axes[0].set_xlabel('Mean DepDelayMinutes')

timeblock_stats.loc[timeblock_order, 'mean_delay'].plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Mean Departure Delay by Time Block (min)')
axes[1].set_ylabel('Mean DepDelayMinutes')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../outputs/figures/delay_by_airline_timeblock.png', dpi=120)
plt.show()

### 3.2 Business-Context Interpretation

> **TODO (fill in after running the cells above):** replace the bracketed placeholders with the actual numbers from `overall_stats`, `airline_stats`, `airport_stats`, and `timeblock_stats`. Keep the sentence structure — the rubric rewards translating a metric into an operational statement, not just restating the number.

**Overall distribution:**
Median `DepDelayMinutes` is `[X]` minutes, but the mean is `[Y]` minutes — the gap between them confirms the distribution is right-skewed: most flights depart close to on time, while a long tail of severely delayed flights pulls the average up. The p90 of `[Z]` minutes means 1 in 10 flights is delayed at least that long, which is the number connection-planning teams should design around, not the mean.

**By airline:**
`[Airline with highest mean_delay]` has the highest mean departure delay (`[X]` min) and the highest `pct_delayed_15` (`[Y]%` of flights delayed 15+ minutes), while `[Airline with lowest mean_delay]` is the most consistent (lowest `std_delay`, `[Z]`). For a traveler or a travel-booking product, this translates into: *choosing `[best airline]` over `[worst airline]` on a comparable route reduces the odds of a 15+ minute delay by roughly `[difference in pct_delayed_15]` percentage points.*

**By airport:**
`[Airport with highest mean_delay]` shows the highest average delay (`[X]` min) despite / because of `[hypothesis: traffic volume, weather exposure, hub congestion — confirm in Section 5]`, whereas `[Airport with lowest mean_delay]` is comparatively efficient. This matters for airline schedule padding and for travelers picking connection hubs.

**By time block:**
Delays are lowest in the `[best time block]` (`[X]` min mean) and highest in the `[worst time block]` (`[Y]` min mean) — consistent with how delays compound through the day as aircraft rotations fall behind schedule (a late inbound aircraft causes a late outbound departure, and the effect accumulates from morning to night). **Actionable takeaway:** an early-morning flight is systematically the lowest-risk choice for a time-sensitive traveler, independent of airline or airport.

## 4. Outlier Detection and Critical Analysis

_Apply IQR or Z-score. Quantify and visualize. Critically evaluate: is the outlier noise or a real signal (storm, mechanical failure)? Decide whether to analyze it separately instead of removing it by default._

## 5. Correlation and Temporal Patterns

_Delay-weather correlation (heatmap). Patterns by time of day, day of week, and season. Identification of "bottleneck" airports._

## 6. Conclusions and Reflection

**Concrete findings:**

**Dataset limitations:**

**Business applications:**

**Ethical considerations:**